# Task 3 — Price Range Analysis

**Objective:** Understand how restaurants are distributed across Zomato's 4 price tiers,  
which price tier offers the best value (highest rating), and how price relates to votes,  
cost, and delivery availability.

**Price Range Key:**
| Range | Label       |
|-------|-------------|
| 1     | Budget      |
| 2     | Mid-Range   |
| 3     | Premium     |
| 4     | Luxury      |

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_cleaning import load_clean
from src.analysis      import (price_range_distribution, price_range_avg_rating,
                                best_price_range, price_delivery_crosstab, PRICE_LABELS)
from src.visualization import (plot_price_range_bar, plot_price_range_pie,
                                plot_price_avg_rating, plot_price_cost_rating)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
df = load_clean()
print(f"{df.shape[0]:,} restaurants | {df['City'].nunique()} cities | Price ranges: {sorted(df['Price range'].unique())}")
df[['Restaurant Name','City','Price range','Aggregate rating','Average Cost for two','Votes']].head(6)

## 2. Price Range Distribution

In [ ]:
price_dist = price_range_distribution(df)
print('Price Range Distribution:')
print(price_dist.to_string(index=False))

In [ ]:
# Bar chart
fig = plot_price_range_bar(price_dist)
plt.show()

In [ ]:
# Pie chart
fig = plot_price_range_pie(price_dist)
plt.show()

## 3. Average Rating per Price Range

In [ ]:
price_stats = price_range_avg_rating(df)
print('Price Range Statistics:')
print(price_stats.to_string(index=False))

best = best_price_range(df)
print(f"\n★ Best price range by avg rating: {best['Label']} (Avg Rating = {best['Avg Rating']})")

In [ ]:
fig = plot_price_avg_rating(price_stats)
plt.show()

## 4. Average Cost vs Rating (Bubble Chart)

In [ ]:
fig = plot_price_cost_rating(price_stats)
plt.show()

## 5. Full Statistics Table

In [ ]:
price_stats

## 6. Box Plot — Rating Distribution per Price Range

In [ ]:
df_plot = df.copy()
df_plot['Price Label'] = df_plot['Price range'].map(PRICE_LABELS)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df_plot, x='Price Label', y='Aggregate rating',
            palette='Set2', width=0.5, order=list(PRICE_LABELS.values()), ax=ax)
ax.set_xlabel('Price Range', fontsize=12)
ax.set_ylabel('Aggregate Rating', fontsize=12)
ax.set_title('Rating Distribution per Price Range', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Price Range × Online Delivery Cross-tab

In [ ]:
ct = price_delivery_crosstab(df)
print('Cross-tab: Price Range × Online Delivery')
ct

In [ ]:
# Stacked bar: delivery vs no delivery within each price range
ct_no_all = ct.drop(index='All', errors='ignore').drop(columns='All', errors='ignore')
ct_no_all.plot(kind='bar', stacked=True, figsize=(9, 5),
               colormap='Set2', edgecolor='white', width=0.55)
plt.xlabel('Price Range', fontsize=12)
plt.ylabel('Number of Restaurants', fontsize=12)
plt.title('Online Delivery Availability by Price Range (Stacked)', fontsize=13, fontweight='bold')
plt.xticks(rotation=15)
plt.legend(title='Delivery Status', fontsize=10)
plt.tight_layout()
plt.show()

## 8. Votes by Price Range

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette('viridis', 4)
bars = ax.bar(price_stats['Label'], price_stats['Avg Votes'],
              color=colors, edgecolor='white', width=0.55)
ax.set_xlabel('Price Range', fontsize=12)
ax.set_ylabel('Average Number of Votes', fontsize=12)
ax.set_title('Average Votes per Price Range', fontsize=14, fontweight='bold')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+5, f'{h:.0f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Business Insights & Conclusions

```
KEY FINDINGS — Task 3: Price Range Analysis
============================================

1. MARKET SPLIT
   Budget (1) and Mid-Range (2) restaurants together represent ~74% of all
   listings. This reflects Zomato's mass-market user base.

2. RATINGS INCREASE WITH PRICE
   There is a clear positive relationship between price tier and average rating:
     Budget (1)    → 3.24
     Mid-Range (2) → 3.38
     Premium (3)   → 3.78  ← notable jump
     Luxury (4)    → 3.89  ← highest rated
   Higher-priced restaurants invest more in quality, staff, and ambience.

3. VOTES PEAK AT PREMIUM (3)
   Premium restaurants get the most votes (~455 avg), even more than Luxury.
   This suggests Premium is the 'sweet spot' — great quality at a price point
   accessible to a broad, engaged audience.

4. BEST VALUE TIER
   Premium (3) offers the best combination of high rating (3.78) AND
   high engagement (455 votes). Worth targeting for marketing campaigns.

5. COST RANGE
   Average cost for two spans from ~₹285 (Budget) to ~₹1,475 (Luxury),
   a 5× difference — confirming the tiers are meaningfully separated.

CUSTOMER RECOMMENDATION
   For the best dining experience within a reasonable budget, Premium (3)
   restaurants deliver the highest ratings AND are well-reviewed.

BUSINESS RECOMMENDATION
   Zomato should feature Premium restaurants more prominently in app
   discovery — they have the highest engagement and user satisfaction.
```